# Load the inference results from disk

In [1]:
import pickle
import os
from tqdm.notebook import tqdm_notebook

def load_inference_results_on_raw_images(inference_results_dir):
    image_files = []
    boxes = []
    words = []
    
    for pkl_file in tqdm_notebook(os.listdir(inference_results_dir)):
        full_path = os.path.join(inference_results_dir, pkl_file)
        with open(full_path, "rb") as f:
            pkl_obj = pickle.load(f)
        image_files.append((pkl_obj["image_file"], len(image_files)))
        boxes.append(pkl_obj["raw_image_box"])
        words.append(pkl_obj["raw_image_word"])
        
    # sort boxes and words based on image_files
    image_files.sort()
    boxes = [boxes[i] for (_,i) in image_files]
    words = [words[i] for (_,i) in image_files]

    return boxes, words

def load_inference_results_on_fully_processed_images(inference_results_dir):
    image_files = []
    boxes = []
    words = []
    
    for pkl_file in tqdm_notebook(os.listdir(inference_results_dir)):
        full_path = os.path.join(inference_results_dir, pkl_file)
        with open(full_path, "rb") as f:
            pkl_obj = pickle.load(f)
        image_files.append((pkl_obj["image_file"], len(image_files)))
        boxes.append(pkl_obj["fully_processed_image_box"])
        words.append(pkl_obj["fully_processed_image_word"])

    # sort boxes and words based on image_files
    image_files.sort()
    boxes = [boxes[i] for (_,i) in image_files]
    words = [words[i] for (_,i) in image_files]

    return boxes, words

In [2]:
raw_image_boxes, raw_image_words = load_inference_results_on_raw_images("../outputs/inference")

  0%|          | 0/549 [00:00<?, ?it/s]

# Evaluating description of Flowcharts of random 3 samples:

In[123]:

In [5]:
import random
random.seed(888)

# Define the size of the sample you want to select
sample_size = 5

# Generate the population range from 0 up to len(raw_image_words)
population = range(len(raw_image_words))

random_numbers = random.sample(population, sample_size)
print("Selected indexes:", random_numbers)

prompts = []

for num in random_numbers:
    x,y = raw_image_words[num], raw_image_boxes[num]
    prompts.append("Give a detailed and descriptive interpretation of the flowchart in the form of steps using following details pytesseract text recognition data:"+str(x)+"bounding box info obtained from sam(segment anything model):"+str(y))

Selected indexes: [81, 442, 455, 508, 395]


In[124]:

In [6]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ.get("OPENROUTER_API_KEY"))

In[199]:

In [9]:
c = 0
answer1 = []
for x in prompts:
    c = c+1
    print(c)
    
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",  # or whichever model you want
        messages=[
            {"role": "system", "content": x}
        ],
        max_tokens=500
    )
    answer1.append(response.choices[0].message.content.replace('\n', '').replace(' .', '.').strip())

1
2
3
4
5


In [10]:
answer1

['Interpretation of the Steps in the Flowchart:1. Patent Sheet:    - Receive a royalty statement related to the license.   - Enter the royalty statement into the system, detailing the period to which it applies and specifics like product units sold, revenue generated, and royalty due.2. Payment Entry:    - Receive a payment related to the license.   - Enter the payment amount into the system.   - Optionally, allocate the payment to terms of the license and link it to related royalty statements.   - Optionally, allocate the payment to details of royalty statements received from the licensee.   3. Reporting:   - Run reports based on needs and interests.   - Examples of reports include comparing payments to royalty statement details, comparing payments to expected revenue, and determining revenues attributable to each asset in a package.4. More Payment Entry:   - Repeat the process of receiving a payment related to the license, entering the payment, and allocating it as in step 2.5. More 

In [11]:
input1 = "Give a detailed and descriptive interpretation of the flowchart in the form of steps using following details pytesseract text recognition data: bounding box info obtained from sam(segment anything model):[[1191, 835, 51, 18], [850, 668, 60, 23], [583, 262, 52, 17], [514, 262, 62, 18], [1345, 835, 43, 18], [1230, 655, 49, 17], [475, 235, 54, 18], [1284, 656, 66, 16], [151, 237, 14, 16], [1286, 540, 14, 65], [84, 10, 264, 175], [84, 10, 264, 121], [85, 11, 262, 510], [84, 10, 263, 260], [0, 0, 1451, 949], [442, 192, 265, 132], [535, 238, 89, 15], [88, 189, 255, 139], [441, 407, 264, 131], [88, 407, 617, 131], [798, 407, 264, 132], [84, 407, 264, 131], [1155, 407, 275, 300], [1155, 407, 275, 132], [797, 611, 617, 132], [797, 612, 264, 132], [1165, 607, 256, 141], [1150, 805, 286, 131]]the text in each of those bounding boxes['', '', '', '', '', '', '', '', '', '', 'Customer places\\n\\nan order\\n', 'Yes\\n', '', 'Customer places\\nan order\\n\\nIs item still in Email customer and\\nstock? cancel order\\n\\nEmail customer with\\nHand off to carrier shipping confirmation\\nand tracking info\\n\\nDoes carrier\\n\\nNotify customer\\n\\ndeliver item?\\n\\nEmail customer with\\n\\nconfirmation of delivery\\nand return instructions\\n', 'Email customer and_\\ncancel order\\n', '', '', 'Print label\\n', 'Pack item Print label\\n', 'Hand off to carrier\\n', 'Pack item\\n', 'Does carrier\\ndeliver item?\\n', '<—No—\\n', '', '', 'Email customer with\\nconfirmation of delivery\\n\\nCoes return ——\\n']"

In[172]:

In [12]:
knowledge1 = '''1.Customer places an order: When a customer initiates an order, this is the starting point of the process. 
2.Is the item still in stock?: A decision box where we check if the item is still in stock if it is pack the item else email the customer and cancel the order
3. Email customer and cancel order: If the item is no longer in stock, this step involves notifying the customer and canceling the order. 
4.Pack item: Once we pack the item , we print the label on the item.
5.Print label: Once we label is printed , we move to the next step handoff to carrier
6.Handoff to carrier: The item is handed over to the carrier for delivery 
7.Email customer with shipping confirmation and tracking info: Assuming the item is in stock, this step involves informing the customer about the shipment with tracking details. 7.Does the carrier deliver the item?: A branching point where the process checks if the carrier delivers the item, if it does, Email customer with confirmation of delivery and return instructions , else notify the customer and cancel the order.'''


In[173]:

In [13]:
input2 = "Give a detailed and descriptive interpretation of the flowchart in the form of steps using following details pytesseract text recognition data: bounding box info obtained from sam(segment anything model):[[42, 164, 17, 6], [27, 89, 32, 8], [95, 158, 14, 6], [166, 169, 17, 6], [129, 83, 4, 5], [59, 205, 10, 6], [160, 83, 19, 7], [129, 169, 33, 8], [26, 176, 28, 6], [46, 218, 10, 10], [143, 84, 3, 7], [58, 120, 14, 6], [97, 72, 11, 6], [104, 73, 4, 5], [46, 131, 10, 11], [56, 164, 3, 6], [12, 14, 3, 6], [66, 207, 3, 4], [111, 168, 11, 10], [58, 176, 17, 6], [129, 83, 17, 8], [111, 82, 10, 10], [46, 203, 10, 26], [39, 77, 23, 8], [12, 14, 22, 8], [46, 45, 10, 11], [162, 84, 4, 5], [68, 121, 4, 5], [97, 72, 5, 5], [42, 164, 4, 6], [59, 205, 5, 6], [96, 81, 25, 11], [96, 86, 14, 1], [46, 31, 10, 25], [51, 31, 0, 14], [46, 117, 10, 26], [51, 117, 0, 14], [51, 203, 0, 14], [51, 31, 0, 14], [51, 117, 0, 14], [95, 158, 5, 6], [57, 238, 19, 8], [51, 203, 0, 14], [63, 121, 9, 5], [8, 5, 86, 25], [8, 5, 86, 40], [8, 5, 86, 110], [122, 74, 64, 25], [6, 57, 89, 59], [6, 45, 105, 86], [6, 57, 180, 59], [0, 0, 191, 261], [124, 161, 64, 24], [6, 144, 89, 58], [6, 144, 89, 84], [19, 173, 78, 82], [19, 230, 64, 25], [19, 218, 64, 37]]the text in each of those bounding boxes['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '[Lamp doesnt work\\n', '', '', '', 'Lamp\\nplunged in?\\n', '', 'Replace bulb\\n', '', 'burned out?\\n\\nRepair lamp\\n', '', '']"

In[174]:

In [14]:
knowledge2 = '''1.Lamp Doesn't Work: In this state , the lamp is not working and move on to check if lamp is plugged in
2. Lamp Plugged in?: A decision point. It checks whether the lamp is plugged in. If it is not then plugin lamp , else check if bulb burned out.
3.Bulb burned out :Check if bulb is burned out , if yes replace bulb else repair lamp'''


In[178]:

In [15]:
prompts = []

for num in random_numbers:
    x,y = raw_image_words[num], raw_image_boxes[num]
    prompts.append("Give a detailed and descriptive interpretation of the flowchart in the form of steps using following details pytesseract text recognition data:"+str(x)+"bounding box info obtained from sam(segment anything model):"+str(y))

In[176]:

In [16]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ.get("OPENROUTER_API_KEY"))

In[204]:

In [17]:
c = 0
answer2 = []
for x in prompts:
    c = c+1
    print(c)
    
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",  # or whichever model you want
        messages=[
            {"role": "system", "content": "You describe flowcharts."},
            {"role": "user", "content":input1},
            {"role": "assistant", "content":knowledge1},
            {"role": "user", "content":input2},
            {"role": "assistant", "content":knowledge2},
            {"role": "user", "content":x}],
        max_tokens=1000
    )
    answer2.append(response.choices[0].message.content.replace('\n', '').replace(' .', '.').strip())

1
2
3
4
5


In [19]:
answer2

['1. Patent: The process starts with a patent-related context, indicating some legal protection or ownership.2. Sheet: Mention of a specific sheet or document related to the patent.3. Receive a Royalty Statement: A step that involves receiving and entering royalty statements into the system, accounting for sales units, revenue generated, and royalties due.4. Receive a Payment: Another step that focuses on receiving and entering payments into the system, including allocations based on various criteria like terms of the license and details of royalty statements.5. Run Reports: A step involving generating reports based on specific needs and interests, like comparing payments to royalty statement details and expected revenue.6. U.S. Patent: Reference to a specific U.S. patent issued on May 24, 2011, as part of the process.7. Fit 2N1R: An additional reference or identifier within the flowchart that may relate to a specific directive or category within the process.',
 '1. **START - Configure